# Figure 3 -- OmegAMP analog generation

In [ ]:
import pandas as pd, numpy as np, re, warnings
from pathlib import Path
from Bio import SeqIO
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'font.size':7, 'axes.titlesize':8, 'axes.labelsize':7,
    'xtick.labelsize':6, 'ytick.labelsize':6, 'legend.fontsize':6,
    'axes.linewidth':0.5, 'axes.edgecolor':'grey',
    'xtick.color':'#555555', 'ytick.color':'#555555',
    'axes.labelcolor':'#333333', 'font.family':'sans-serif',
    'font.sans-serif':['Arial','DejaVu Sans'],
    'figure.dpi':300, 'savefig.dpi':300,
    'xtick.major.width':0.6, 'ytick.major.width':0.6,
    'xtick.major.size':2, 'ytick.major.size':2,
    'axes.spines.top':True, 'axes.spines.right':True,
    'figure.facecolor':'white', 'axes.facecolor':'white',
})

ROOT = Path("../data")
SWEEP = ROOT / "figure3/sweep"
FLAVORS = ROOT / "figure3/flavors"
TAU_VALUES = [0, 50, 100, 150, 200, 300, 500, 700]
SIGMA_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0]

import sys, os
sys.path.insert(0, os.path.abspath('..'))
from amp_similarity import normalized_alignment_score

# == PANEL A: Property MAE from sweep (using modlamp for correct properties) ==
print("Panel A: MAE...")
import modlamp.analysis as manalysis

def batch_charge(seqs):
    h = manalysis.GlobalAnalysis(list(seqs)); h.calc_charge(); return h.charge[0]
def batch_hydro(seqs):
    h = manalysis.GlobalAnalysis(list(seqs)); h.calc_H(scale='eisenberg'); return h.H[0]

mae_rows = []
for pt in ['positive','negative']:
    for tau in TAU_VALUES:
        for sigma in SIGMA_VALUES:
            fasta_f = SWEEP / pt / "sequences" / f"{pt}_tau{tau}_sigma{sigma}.fasta"
            if not fasta_f.exists(): continue
            records = list(SeqIO.parse(fasta_f, 'fasta'))
            analogs, protos = [], []
            for r in records:
                parts = r.id.split('|')
                if len(parts) < 2: continue
                analogs.append(str(r.seq))
                clean_proto = ''.join(c for c in parts[1] if c in 'ACDEFGHIKLMNPQRSTVWY')
                protos.append(clean_proto)
            if not analogs: continue
            a_charge = batch_charge(analogs); p_charge = batch_charge(protos)
            a_hydro = batch_hydro(analogs); p_hydro = batch_hydro(protos)
            a_len = np.array([len(s) for s in analogs]); p_len = np.array([len(s) for s in protos])
            for i in range(len(analogs)):
                mae_rows.append({
                    'pt':pt,'tau':tau,'sigma':sigma,
                    'dc':abs(a_charge[i]-p_charge[i]),
                    'dl':abs(a_len[i]-p_len[i]),
                    'dh':abs(a_hydro[i]-p_hydro[i]),
                })
mae_df = pd.DataFrame(mae_rows)
mae_agg = mae_df.groupby(['tau','sigma']).agg(
    dc_mean=('dc','mean'), dc_std=('dc','std'),
    dl_mean=('dl','mean'), dl_std=('dl','std'),
    dh_mean=('dh','mean'), dh_std=('dh','std'),
).reset_index()
print(f"  {len(mae_df)} total rows, {len(mae_agg)} aggregated")
t0 = mae_agg[(mae_agg['tau']==0)&(mae_agg['sigma']==0.0)]
if len(t0)>0: print(f"  tau=0,sigma=0: dC={t0['dc_mean'].values[0]:.4f}, dL={t0['dl_mean'].values[0]:.4f}, dH={t0['dh_mean'].values[0]:.6f}")

# == PANEL B: Pareto from pre-computed AMP-alignment similarity ==
print("Panel B: Pareto (AMP-alignment)...")
pareto_csv = Path("../data/figure3/precomputed/pareto_amp_alignment.csv")
if pareto_csv.exists():
    pareto_df = pd.read_csv(pareto_csv)
    print(f"  Loaded {len(pareto_df)} points from {pareto_csv.name}")
else:
    print("  WARNING: pareto_amp_alignment.csv not found, computing from FASTA...")
    pareto_rows = []
    for pt in ['positive','negative']:
        for tau in TAU_VALUES:
            for sigma in SIGMA_VALUES:
                pred_f = SWEEP / pt / "predictions" / f"{pt}_tau{tau}_sigma{sigma}-min.tsv"
                fasta_f = SWEEP / pt / "sequences" / f"{pt}_tau{tau}_sigma{sigma}.fasta"
                if not pred_f.exists() or not fasta_f.exists(): continue
                proto_map = {}
                for r in SeqIO.parse(fasta_f, 'fasta'):
                    parts = r.id.split('|')
                    if len(parts) < 2: continue
                    m = re.search(r'prototype_(\d+)', r.id)
                    pid = int(m.group(1)) if m else -1
                    proto_map[r.id] = (str(r.seq), parts[1], pid)
                pdf = pd.read_csv(pred_f, sep='\t')
                pdf['proto_id'] = pdf['Sequence_id'].apply(
                    lambda x: int(re.search(r'prototype_(\d+)', x).group(1)) if re.search(r'prototype_(\d+)', x) else -1)
                best = pdf.loc[pdf.groupby('proto_id')['MIC'].idxmin()]
                mic_vals, sim_vals = [], []
                for _, row in best.iterrows():
                    sid = row['Sequence_id']
                    if sid in proto_map:
                        aseq, pseq, _ = proto_map[sid]
                        sim = normalized_alignment_score(aseq, pseq)
                        mic_vals.append(row['MIC']); sim_vals.append(sim)
                if mic_vals:
                    pareto_rows.append({'pt':pt,'tau':tau,'sigma':sigma,
                        'mic':np.median(np.log2(np.array(mic_vals))),'sim':np.median(sim_vals)})
    pareto_df = pd.DataFrame(pareto_rows)
    print(f"  Computed {len(pareto_df)} points")

# == PANELS C,D: Flavors data ==
print("Panels C,D: Flavors...")
def load_flavors():
    mic_rows, sim_rows = [], []
    F = FLAVORS
    MIC_MAP = [
        ('0_omegamp_baselines/predictions/OmegAMP-U-min.tsv', 'OmegAMP-U', None),
        ('0_omegamp_baselines/predictions/OmegAMP-T-min.tsv', 'OmegAMP-T', None),
        ('1_prototypes/predictions/{pt}_prototypes-min.tsv', 'Prototypes', True),
        ('2_analog_only/predictions/{pt}_analog_only-min.tsv', 'OmegAMP-A', True),
        ('3_analog_property/predictions/{pt}_analog_property-min.tsv', 'OmegAMP-AT', True),
        ('4_analog_template/predictions/{pt}_analog_template-min.tsv', 'OmegAMP-AM', True),
        ('5_analog_template_property/predictions/{pt}_analog_template_property-min.tsv', 'OmegAMP-AMT', True),
        ('0_hydramp/predictions/unconditional-min.tsv', 'HydrAMP-U', None),
        ('0_hydramp/predictions/{pt}_1-min.tsv', 'HydrAMP-A\nτ=1', True),
        ('0_hydramp/predictions/{pt}_2.5-min.tsv', 'HydrAMP-A\nτ=2.5', True),
        ('0_hydramp/predictions/{pt}_5-min.tsv', 'HydrAMP-A\nτ=5', True),
    ]
    SIM_MAP = [
        ('0_omegamp_baselines/similarity/OmegAMP-U_vs_curated-AMPs_samples.csv',
         '0_omegamp_baselines/similarity/OmegAMP-U_vs_curated-Non-AMPs_samples.csv', 'OmegAMP-U'),
        ('0_omegamp_baselines/similarity/OmegAMP-T_vs_curated-AMPs_samples.csv',
         '0_omegamp_baselines/similarity/OmegAMP-T_vs_curated-Non-AMPs_samples.csv', 'OmegAMP-T'),
        ('1_prototypes/similarity/positive_prototypes_vs_curated-AMPs_samples.csv',
         '1_prototypes/similarity/negative_prototypes_vs_curated-Non-AMPs_samples.csv', 'Prototypes'),
        ('2_analog_only/similarity/positive_analog_only_vs_positive_prototypes.csv',
         '2_analog_only/similarity/negative_analog_only_vs_negative_prototypes.csv', 'OmegAMP-A'),
        ('3_analog_property/similarity/positive_analog_property_vs_curated-AMPs_samples.csv',
         '3_analog_property/similarity/negative_analog_property_vs_curated-Non-AMPs_samples.csv', 'OmegAMP-AT'),
        ('4_analog_template/similarity/positive_analog_template_vs_curated-AMPs_samples.csv',
         '4_analog_template/similarity/negative_analog_template_vs_curated-Non-AMPs_samples.csv', 'OmegAMP-AM'),
        ('5_analog_template_property/similarity/positive_analog_template_property_vs_curated-AMPs_samples.csv',
         '5_analog_template_property/similarity/negative_analog_template_property_vs_curated-Non-AMPs_samples.csv', 'OmegAMP-AMT'),
        ('0_hydramp/similarity/positive_1_vs_positive_prototypes.csv',
         '0_hydramp/similarity/negative_1_vs_negative_prototypes.csv', 'HydrAMP-A\nτ=1'),
        ('0_hydramp/similarity/positive_2.5_vs_positive_prototypes.csv',
         '0_hydramp/similarity/negative_2.5_vs_negative_prototypes.csv', 'HydrAMP-A\nτ=2.5'),
        ('0_hydramp/similarity/positive_5_vs_positive_prototypes.csv',
         '0_hydramp/similarity/negative_5_vs_negative_prototypes.csv', 'HydrAMP-A\nτ=5'),
    ]
    def load_mic(path_tpl, method, has_pt):
        for pt in (['positive','negative'] if has_pt else ['positive']):
            fp = F / (path_tpl.format(pt=pt) if has_pt else path_tpl)
            if not fp.exists(): continue
            pdf = pd.read_csv(fp, sep='\t')
            targets = [pt] if has_pt else ['positive','negative']
            for t in targets:
                for _, row in pdf.iterrows(): mic_rows.append({'method':method,'pt':t,'mic':row['MIC']})
    def load_sim(pos_path, neg_path, method):
        for pt, path in [('positive', pos_path), ('negative', neg_path)]:
            fp = F / path
            if not fp.exists(): continue
            sdf = pd.read_csv(fp)
            best = sdf.groupby('queryId')['normalized'].max()
            for v in best.values: sim_rows.append({'method':method,'pt':pt,'sim':v})
    for path, method, has_pt in MIC_MAP: load_mic(path, method, has_pt)
    for pos_p, neg_p, method in SIM_MAP: load_sim(pos_p, neg_p, method)
    return pd.DataFrame(mic_rows), pd.DataFrame(sim_rows)

fl_mic, fl_sim = load_flavors()
NAME_MAP = {'OmegAMP-U':'OmegAMP-DU','OmegAMP-T':'OmegAMP-DT','OmegAMP-A':'OmegAMP-AU'}
fl_mic['method'] = fl_mic['method'].replace(NAME_MAP)
fl_sim['method'] = fl_sim['method'].replace(NAME_MAP)
print(f"  MIC: {len(fl_mic)} rows, methods: {sorted(fl_mic['method'].unique())}")
print(f"  Sim: {len(fl_sim)} rows, methods: {sorted(fl_sim['method'].unique())}")

# ================================================================
# PLOTTING
# ================================================================
fig = plt.figure(figsize=(6.5, 8.0), dpi=300)
A_TOP, A_BOT = 0.98, 0.80
B_TOP, B_BOT = 0.76, 0.58
CD_TOP, CD_BOT = 0.47, 0.07
LABEL_X = 0.01
gs_a = gridspec.GridSpec(1,3,figure=fig,left=0.09,right=0.97,top=A_TOP,bottom=A_BOT,wspace=0.35)
gs_b = gridspec.GridSpec(1,2,figure=fig,left=0.09,right=0.97,top=B_TOP,bottom=B_BOT,wspace=0.20)
gs_cd = gridspec.GridSpec(2,2,figure=fig,left=0.09,right=0.97,top=CD_TOP,bottom=CD_BOT,
                          wspace=0.12,hspace=0.12,height_ratios=[1,1])
PL = dict(fontsize=10, fontweight='bold', va='top', ha='left')
fig.text(LABEL_X, A_TOP, 'A', **PL)
fig.text(LABEL_X, B_TOP, 'B', **PL)
fig.text(LABEL_X, CD_TOP, 'C', **PL)
fig.text(LABEL_X, (CD_TOP+CD_BOT)/2+0.01, 'D', **PL)

TAU_PLOT = [0,100,200,300,500,700]
TAU_COLORS = {0:'#440154',100:'#365c8d',200:'#1fa187',300:'#4ac16d',500:'#9fda3a',700:'#fde725'}
SIGMA_COLORS = {0.0:'#fee5d9',0.25:'#fcae91',0.5:'#fb6a4a',0.75:'#de2d26',1.0:'#a50f15'}

# -- A: Property MAE --
for pi,(prop,ylabel) in enumerate([('dc','\u0394C'),('dl','\u0394L'),('dh','\u0394H')]):
    ax = fig.add_subplot(gs_a[pi])
    for tau in TAU_PLOT:
        td = mae_agg[mae_agg['tau']==tau].sort_values('sigma')
        if len(td)==0: continue
        ax.errorbar(td['sigma'],td[f'{prop}_mean'],yerr=td[f'{prop}_std'],fmt='o-',
                   color=TAU_COLORS[tau],ms=3,lw=1,elinewidth=0.5,capsize=1.5,capthick=0.5,
                   label=f'{tau/1000:.1f}' if pi==2 else None)
    ax.set_xlabel('\u03C3',fontsize=6); ax.set_ylabel(ylabel,fontsize=7)
    for spine in ax.spines.values(): spine.set_edgecolor('grey'); spine.set_linewidth(0.5)
    if pi==2:
        ax.legend(title='\u03C4',fontsize=6,title_fontsize=6,loc='upper left',frameon=True,
                 fancybox=True,edgecolor='#ccc',ncol=2,handlelength=1,columnspacing=0.5)

# -- B: Pareto front --
MIC_THRESH = 5; SIM_THRESH = 60
TAU_SIZES = {0:6,50:10,100:14,150:18,200:22,300:28,500:36,700:44}

# Parameters used for analog generation in panels C/D (from scripts)
USED_TAU = {'positive': 150, 'negative': 100}
USED_SIGMA = {'positive': 1.0, 'negative': 1.0}

for pi, pt in enumerate(['positive','negative']):
    ax = fig.add_subplot(gs_b[pi])
    title = 'Active' if pt=='positive' else 'Inactive'
    ax.text(0.03,0.97,title,transform=ax.transAxes,fontsize=8,fontweight='bold',va='top')

    pdf = pareto_df[pareto_df['pt']==pt]

    ax.fill_betweenx([1.5, MIC_THRESH], SIM_THRESH, 108, alpha=0.10, color='#90EE90', zorder=0)
    ax.axvline(SIM_THRESH, color='#aaddaa', ls='--', lw=0.5, zorder=1)
    ax.axhline(MIC_THRESH, color='#aaddaa', ls=':', lw=0.5, zorder=1)

    for sigma in SIGMA_VALUES:
        sdf = pdf[pdf['sigma']==sigma].sort_values('tau')
        if len(sdf)>1:
            ax.plot(sdf['sim'], sdf['mic'], color=SIGMA_COLORS[sigma], lw=0.8, alpha=0.6, zorder=2)

    for _, row in pdf.iterrows():
        ax.scatter(row['sim'], row['mic'], c=SIGMA_COLORS[row['sigma']],
                  s=TAU_SIZES.get(int(row['tau']),15), edgecolors='#333', linewidth=0.3, zorder=3, alpha=0.85)

    # Highlight the parameters used for generating analogs in C/D
    sel = pdf[(pdf['tau']==USED_TAU[pt])&(pdf['sigma']==USED_SIGMA[pt])]
    if len(sel)>0:
        best = sel.iloc[0]
        ax.scatter(best['sim'],best['mic'],s=55,facecolors='none',edgecolors='red',linewidth=1.5,zorder=5)
        ax.annotate(f"\u03C4={best['tau']/1000}, \u03C3={best['sigma']}",
                   (best['sim'],best['mic']),fontsize=6,fontweight='bold',color='#b22222',
                   xytext=(5,-15),textcoords='offset points')

    ax.set_xlabel('Sequence similarity (%)',fontsize=6)
    ax.set_ylabel(r'log$_2$ $MIC_{APEX}$ ($\mu$M)',fontsize=6)
    ax.set_xlim(-5, 108); ax.set_ylim(1.5, 6.2)
    for spine in ax.spines.values(): spine.set_edgecolor('grey'); spine.set_linewidth(0.5)

# B legends
sigma_handles = [Line2D([],[],marker='o',color='w',markerfacecolor=SIGMA_COLORS[s],markeredgecolor='#333',
                        ms=4,lw=0,label=f'{s}') for s in SIGMA_VALUES]
tau_handles = [Line2D([],[],marker='o',color='w',markerfacecolor='#888',markeredgecolor='#333',
                      ms=np.sqrt(TAU_SIZES[t])/1.5,lw=0,label=f'{t/1000:.0f}' if t==0 else f'{t/1000:.1f}')
               for t in [0,200,700]]
fig.legend(handles=sigma_handles,title='\u03C3',loc='upper center',bbox_to_anchor=(0.29,0.54),
          ncol=5,fontsize=6,title_fontsize=6,frameon=True,edgecolor='#ccc',
          fancybox=True,handletextpad=0.1,columnspacing=0.4)
fig.legend(handles=tau_handles,title='\u03C4',loc='upper center',bbox_to_anchor=(0.77,0.54),
          ncol=3,fontsize=6,title_fontsize=6,frameon=True,edgecolor='#ccc',
          fancybox=True,handletextpad=0.1,columnspacing=0.4)

# -- C+D: Combined violin panels (2x2: MIC top, Similarity bottom) --
METHOD_ORDER = ['OmegAMP-DU','OmegAMP-DT','Prototypes','OmegAMP-AU','OmegAMP-AT','OmegAMP-AM','OmegAMP-AMT',
                'HydrAMP-U','HydrAMP-A\n\u03C4=1','HydrAMP-A\n\u03C4=2.5','HydrAMP-A\n\u03C4=5']
GRP_COLORS = {
    'OmegAMP-DU':'#9B8EC4','OmegAMP-DT':'#6B5B95','Prototypes':'#808080',
    'OmegAMP-AU':'#D4B0B0','OmegAMP-AT':'#C9A0A0','OmegAMP-AM':'#A35D5D','OmegAMP-AMT':'#7B2D3D',
    'HydrAMP-U':'#A8DCD9','HydrAMP-A\n\u03C4=1':'#7FCAC3','HydrAMP-A\n\u03C4=2.5':'#3A9E95','HydrAMP-A\n\u03C4=5':'#1A5F5A',
}
SEP_POSITIONS = [1.5, 2.5, 6.5]

avail_mic = [m for m in METHOD_ORDER if m in fl_mic['method'].unique()]
avail_sim = [m for m in METHOD_ORDER if m in fl_sim['method'].unique()]
n_methods = len(avail_mic)

def violin_panel(ax, data, val_col, pt, methods, colors, threshold=None, log=False, ylim=None, show_xlabels=False, show_n=True):
    positions, vdata, cols, ns = [], [], [], []
    for i,m in enumerate(methods):
        d = data[(data['method']==m)&(data['pt']==pt)][val_col].dropna().values
        if len(d)==0: positions.append(i); vdata.append(np.array([])); cols.append('#ccc'); ns.append(0); continue
        if log: d = np.log2(np.clip(d, 0.5, None))
        positions.append(i); vdata.append(d); cols.append(colors.get(m,'#999')); ns.append(len(d))
    valid = [(p,d,c) for p,d,c in zip(positions,vdata,cols) if len(d)>0]
    if not valid: return ns
    vpos, vd, vc = zip(*valid)
    parts = ax.violinplot(vd, positions=vpos, widths=0.75, showmedians=False, showextrema=False)
    for pc, c in zip(parts['bodies'], vc):
        pc.set_facecolor(c); pc.set_edgecolor('none'); pc.set_alpha(0.65)
    for pos, d in zip(vpos, vd):
        q1, med, q3 = np.percentile(d, [25,50,75])
        ax.vlines(pos, q1, q3, color='#333', lw=1.5, zorder=4)
        ax.scatter([pos], [med], color='white', s=6, zorder=5, edgecolors='#333', linewidth=0.5)
    if threshold is not None:
        ax.axhline(np.log2(threshold) if log else threshold, color='#E8912D', ls='--', lw=0.8, zorder=1)
    for sx in SEP_POSITIONS: ax.axvline(sx, color='#ccc', ls=':', lw=0.5, zorder=0)
    ax.set_xticks(range(len(methods)))
    if show_xlabels:
        ax.set_xticklabels(methods, fontsize=6, rotation=45, ha='right')
    else:
        ax.set_xticklabels([])
    if ylim: ax.set_ylim(ylim)
    for spine in ax.spines.values(): spine.set_edgecolor('grey'); spine.set_linewidth(0.5)
    return ns

# Top row: MIC (C)
for pi, pt in enumerate(['positive','negative']):
    ax = fig.add_subplot(gs_cd[0, pi])
    ns = violin_panel(ax, fl_mic, 'mic', pt, avail_mic, GRP_COLORS, threshold=32, log=True, ylim=(0,7.5), show_xlabels=False)
    ax.set_ylabel(r'log$_2$ $MIC_{APEX}$ ($\mu$M)' if pi==0 else '',fontsize=6)
    ymax = ax.get_ylim()[1]
    for i, n in enumerate(ns):
        if n==0: continue
        label = f'n={n//1000}k' if n>=1000 else f'n={n}'
        ax.text(i, ymax*1.01, label, ha='center', va='bottom', fontsize=6, color='#888')

# Bottom row: Similarity (D)
for pi, pt in enumerate(['positive','negative']):
    ax = fig.add_subplot(gs_cd[1, pi])
    ns = violin_panel(ax, fl_sim, 'sim', pt, avail_sim, GRP_COLORS, threshold=0.6, log=False, ylim=(0,1.05), show_xlabels=True, show_n=False)
    ax.set_ylabel('Sequence Similarity' if pi==0 else '',fontsize=6)

# Bottom legend
grp_handles = [
    Patch(facecolor='#9B8EC4',edgecolor='none',label='OmegAMP-D'),
    Patch(facecolor='#808080',edgecolor='none',label='Prototypes'),
    Patch(facecolor='#A35D5D',edgecolor='none',label='OmegAMP-A'),
    Patch(facecolor='#3A9E95',edgecolor='none',label='HydrAMP'),
    Line2D([],[],ls='--',color='#E8912D',lw=0.8,label='Threshold'),
]
fig.legend(handles=grp_handles,loc='lower center',bbox_to_anchor=(0.52,-0.02),
          ncol=5,fontsize=6,frameon=False,handletextpad=0.3,columnspacing=1)

plt.savefig('../figures/figure3_analog.pdf', bbox_inches='tight')
plt.savefig('../figures/figure3_analog.png', bbox_inches='tight')
plt.show()
print("Saved figure3_analog")